In [1]:
# Ignore pre-production warnings
import warnings
warnings.filterwarnings('ignore')

import nemo
# Import Speech Recognition collection
import nemo.collections.asr as nemo_asr
# Import Speech Synthesis collection
import nemo.collections.tts as nemo_tts
# We'll use this to listen to audio
import IPython

fused_indices_to_multihot has reached end of life. Please migrate to a non-experimental function.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


In [3]:
# Download audio sample which we'll try
# This is a sample from LibriSpeech Dev Clean dataset - the model hasn't seen it before
Audio_sample = '2086-149220-0033.wav'
!wget https://dldata-public.s3.us-east-2.amazonaws.com/2086-149220-0033.wav
# Listen to it
IPython.display.Audio(Audio_sample)

--2025-12-27 13:52:27--  https://dldata-public.s3.us-east-2.amazonaws.com/2086-149220-0033.wav
Resolving dldata-public.s3.us-east-2.amazonaws.com (dldata-public.s3.us-east-2.amazonaws.com)... 3.5.130.155, 52.219.94.226, 52.219.142.42, ...
Connecting to dldata-public.s3.us-east-2.amazonaws.com (dldata-public.s3.us-east-2.amazonaws.com)|3.5.130.155|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 237964 (232K) [audio/wav]
Saving to: ‘2086-149220-0033.wav’

2086-149220-0033.wa 100%[===================>] 232.39K   261KB/s    in 0.9s    

2025-12-27 13:52:29 (261 KB/s) - ‘2086-149220-0033.wav’ saved [237964/237964]



In [4]:

# Speech Recognition model - QuartzNet
quartznet = nemo_asr.models.EncDecCTCModel.from_pretrained(model_name="stt_en_quartznet15x5").cuda()

# Spectrogram generator which takes text as an input and produces spectrogram
spectrogram_generator = nemo_tts.models.FastPitchModel.from_pretrained(model_name="tts_en_fastpitch").cuda()

# Vocoder model which takes spectrogram and produces actual audio
vocoder = nemo_tts.models.HifiGanModel.from_pretrained(model_name="tts_en_hifigan").cuda()

[NeMo I 2025-12-27 13:53:22 nemo_logging:393] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/stt_en_quartznet15x5/versions/1.0.0rc1/files/stt_en_quartznet15x5.nemo to /home/raco/.cache/torch/NeMo/NeMo_2.6.0/stt_en_quartznet15x5/16661021d16e679bdfd97a2a03944c49/stt_en_quartznet15x5.nemo
100% [........................................................................] 70993538 / 70993538[NeMo I 2025-12-27 13:53:30 nemo_logging:393] Instantiating model from pre-trained checkpoint


[NeMo W 2025-12-27 13:53:30 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data2/voices/train_1k.json
    sample_rate: 16000
    labels:
    - ' '
    - a
    - b
    - c
    - d
    - e
    - f
    - g
    - h
    - i
    - j
    - k
    - l
    - m
    - 'n'
    - o
    - p
    - q
    - r
    - s
    - t
    - u
    - v
    - w
    - x
    - 'y'
    - z
    - ''''
    batch_size: 32
    trim_silence: true
    max_duration: 16.7
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: /asr_set_1.2/train/train_{0..1023}.tar
    num_workers: 20
    
[NeMo W 2025-12-27 13:53:30 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loade

[NeMo I 2025-12-27 13:53:30 nemo_logging:393] PADDING: 16
[NeMo I 2025-12-27 13:53:31 nemo_logging:393] Model EncDecCTCModel was successfully restored from /home/raco/.cache/torch/NeMo/NeMo_2.6.0/stt_en_quartznet15x5/16661021d16e679bdfd97a2a03944c49/stt_en_quartznet15x5.nemo.
[NeMo I 2025-12-27 13:53:31 nemo_logging:393] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/tts_en_fastpitch/versions/1.8.1/files/tts_en_fastpitch_align.nemo to /home/raco/.cache/torch/NeMo/NeMo_2.6.0/tts_en_fastpitch_align/b7d086a07b5126c12d5077d9a641a38c/tts_en_fastpitch_align.nemo
100% [......................................................................] 187023360 / 187023360[NeMo I 2025-12-27 13:53:51 nemo_logging:393] Instantiating model from pre-trained checkpoint


 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars.
[NeMo W 2025-12-27 13:54:02 nemo_logging:405] apply_to_oov_word=None, This means that some of words will remain unchanged if they are not handled by any of the rules in self.parse_one_word(). This may be intended if phonemes and chars are both valid inputs, otherwise, you may see unexpected deletions in your input.
[NeMo W 2025-12-27 13:54:02 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    dataset:
      _target_: nemo.collections.tts.torch.data.TTSDataset
      manifest_filepath: /ws/LJSpeech/nvidia_ljspeech_train_clean_ngc.json
      sample_rate: 22050
      sup_data_path: /raid/LJSpeech/supplementary
      sup_data_types:
      - align_prior_matrix
      - pitch
      n_fft: 1024
      win_length: 1024
      hop_length: 256
      window: hann
      n_mels:

[NeMo I 2025-12-27 13:54:02 nemo_logging:393] PADDING: 1
[NeMo I 2025-12-27 13:54:02 nemo_logging:393] Model FastPitchModel was successfully restored from /home/raco/.cache/torch/NeMo/NeMo_2.6.0/tts_en_fastpitch_align/b7d086a07b5126c12d5077d9a641a38c/tts_en_fastpitch_align.nemo.
[NeMo I 2025-12-27 13:54:02 nemo_logging:393] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/tts_hifigan/versions/1.0.0rc1/files/tts_hifigan.nemo to /home/raco/.cache/torch/NeMo/NeMo_2.6.0/tts_hifigan/e6da322f0f7e7dcf3f1900a9229a7e69/tts_hifigan.nemo
100% [......................................................................] 315386678 / 315386678[NeMo I 2025-12-27 13:54:32 nemo_logging:393] Instantiating model from pre-trained checkpoint


[NeMo W 2025-12-27 13:54:34 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    dataset:
      _target_: nemo.collections.tts.data.datalayers.MelAudioDataset
      manifest_filepath: /home/fkreuk/data/train_finetune.txt
      min_duration: 0.75
      n_segments: 8192
    dataloader_params:
      drop_last: false
      shuffle: true
      batch_size: 64
      num_workers: 4
    
[NeMo W 2025-12-27 13:54:34 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    dataset:
      _target_: nemo.collections.tts.data.datalayers.MelAudioDataset
      manifest_filepath: /home/fkreuk/data/val_finetune.txt
      min_duration: 3
      n_segmen

[NeMo I 2025-12-27 13:54:34 nemo_logging:393] PADDING: 0


[NeMo W 2025-12-27 13:54:34 nemo_logging:405] Using torch_stft is deprecated and has been removed. The values have been forcibly set to False for FilterbankFeatures and AudioToMelSpectrogramPreprocessor. Please set exact_pad to True as needed.


[NeMo I 2025-12-27 13:54:34 nemo_logging:393] PADDING: 0
[NeMo I 2025-12-27 13:54:35 nemo_logging:393] Model HifiGanModel was successfully restored from /home/raco/.cache/torch/NeMo/NeMo_2.6.0/tts_hifigan/e6da322f0f7e7dcf3f1900a9229a7e69/tts_hifigan.nemo.


In [5]:

# Convert our audio sample to text
files = [Audio_sample]
raw_text = ''
text = ''
for fname, transcription in zip(files, quartznet.transcribe(audio=files)):
  raw_text = transcription

print(f'\nRaw recognized text: {raw_text}')

Transcribing: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.94it/s]


Raw recognized text: Hypothesis(score=tensor(-7.0592), y_sequence=tensor([28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28,
        28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28,
        28, 23, 28,  5, 12, 28, 12, 12, 28, 28,  0,  0,  0,  0,  9,  0,  0,  0,
         0,  4, 15, 28, 14, 27, 20, 20, 28,  0,  0, 28, 23,  9, 28, 19, 28,  8,
        28, 28,  0,  0,  0, 20, 15,  0,  0,  0,  0, 19,  5, 28,  5, 28,  0,  0,
         9, 20,  0,  0,  0,  0,  0, 28,  1, 14, 28, 28, 25, 28, 28,  0,  0, 13,
        28, 15, 28, 18,  5, 28,  0,  0,  0,  0,  0,  0,  0,  0, 28, 15,  2, 28,
        28, 28, 28, 19, 28,  5, 28, 18, 22,  5,  5,  4, 28,  0,  0, 28, 28, 28,
        16,  8, 15,  5, 28, 28, 28,  2, 28,  5, 28, 28, 28, 28, 28, 28, 28, 28,
        28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28,  0,  0, 28, 28, 20,
        28, 21, 28, 18, 28, 14, 14, 28,  9, 14,  7, 28,  0,  0, 28,  1, 28, 28,
        23, 28,  1, 25, 28,  0,  0,  0,  0,  0,  0,  

In [7]:
# A helper function which combines TTS models to go directly from 
# text to audio
def text_to_audio(text):
  parsed = spectrogram_generator.parse(text)
  spectrogram = spectrogram_generator.generate_spectrogram(tokens=parsed)
  audio = vocoder.convert_spectrogram_to_audio(spec=spectrogram)
  return audio.to('cpu').detach().numpy()

In [8]:
# This is our original audio sample
IPython.display.Audio(Audio_sample)

In [9]:

# This is what was recognized by the ASR model
print(raw_text)


Hypothesis(score=tensor(-7.0592), y_sequence=tensor([28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28,
        28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28,
        28, 23, 28,  5, 12, 28, 12, 12, 28, 28,  0,  0,  0,  0,  9,  0,  0,  0,
         0,  4, 15, 28, 14, 27, 20, 20, 28,  0,  0, 28, 23,  9, 28, 19, 28,  8,
        28, 28,  0,  0,  0, 20, 15,  0,  0,  0,  0, 19,  5, 28,  5, 28,  0,  0,
         9, 20,  0,  0,  0,  0,  0, 28,  1, 14, 28, 28, 25, 28, 28,  0,  0, 13,
        28, 15, 28, 18,  5, 28,  0,  0,  0,  0,  0,  0,  0,  0, 28, 15,  2, 28,
        28, 28, 28, 19, 28,  5, 28, 18, 22,  5,  5,  4, 28,  0,  0, 28, 28, 28,
        16,  8, 15,  5, 28, 28, 28,  2, 28,  5, 28, 28, 28, 28, 28, 28, 28, 28,
        28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28, 28,  0,  0, 28, 28, 20,
        28, 21, 28, 18, 28, 14, 14, 28,  9, 14,  7, 28,  0,  0, 28,  1, 28, 28,
        23, 28,  1, 25, 28,  0,  0,  0,  0,  0,  0,  8,  5, 18, 18,  0,  0,

In [10]:
# Without punctuation
IPython.display.Audio(text_to_audio(raw_text), rate=22050)

[NeMo W 2025-12-27 13:59:00 nemo_logging:405] parse() is meant to be called in eval mode.
[NeMo W 2025-12-27 13:59:00 nemo_logging:405] generate_spectrogram() is meant to be called in eval mode.
